In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
from plotnine import *

import requests
import json
import re
import sys
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option("display.max_columns", None)

In [9]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`

In [19]:
def get_model_id(url):
    """Extracts the model ID (number after modeldb.science/) from a given ModelDB URL."""
    match = re.search(r'modeldb\.science/(\d+)', url)
    return match.group(1) if match else None

def get_pubmed_link(model_id):
    if model_id is None:
        return "Invalid model_id"

    url = f"https://modeldb.science/{model_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        return f"Error: {e}"

    soup = BeautifulSoup(response.text, "html.parser")

    # Look for PubMed links
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if "ncbi.nlm.nih.gov/pubmed" in href:
            # Extract PubMed ID if possible
            match = re.search(r"term=(\d+)", href)
            if match:
                pubmed_id = match.group(1)
                return f"https://pubmed.ncbi.nlm.nih.gov/{pubmed_id}/"
            return href  # fallback to raw URL

    return "No PubMed link found"


In [38]:
json_df = pd.read_json("../data/mod_files.json")
df_pre = pd.read_csv("../code/preprocessed_orig.csv")

In [39]:
ANNOTATED_FILES = list(df_pre["file_hash"])

In [48]:
from tqdm import tqdm
tqdm.pandas()  # enables `progress_apply`

json_df_sample = (
    json_df
    .query("file_hash in @ANNOTATED_FILES")
    .reset_index(drop=True)
    .assign(
        model_id=lambda df: df['url'].apply(get_model_id),
        pubmed_link=lambda df: df['model_id'].progress_apply(get_pubmed_link)
    )
)


100%|██████████| 737/737 [00:59<00:00, 12.42it/s]


In [50]:
json_df_sample.query("pubmed_link == 'No PubMed link found'")

,row_id,file_hash,raw_sha,count,url,download_url,content,error_code,model_id,pubmed_link
6,12,b9649c41844bcfe953ae41702b70b74e97b0ed5f296693...,61378102dbdfee7f5af0adfa35f08c1d20048c01e66f36...,2,https://modeldb.science/2014814?tab=2&file=201...,https://modeldb.science/getModelFile?model=201...,: Documentation: https://github.com/fietkiewic...,None,2014814,No PubMed link found
23,34,8eb4d734fab7de07267a3049d84112f949e19679e75026...,3621c69ea12e67ba930ba20a0385078b86725b3f4ed86a...,1,https://modeldb.science/267508?tab=2&file=MSN(...,https://modeldb.science/getModelFile?model=267...,TITLE transient and low threshold calcium curr...,None,267508,No PubMed link found
36,57,bba6529126b4fcb09e495c2b5b47ae79e0e4275e143bd4...,b2566141171bc8da887c922f43b4179c694bb36848bd8d...,1,https://modeldb.science/87473?tab=2&file=weave...,https://modeldb.science/getModelFile?model=874...,": width = yvec.width(xvec, yval) sensitive to...",None,87473,No PubMed link found
44,68,5ab3a41f600df73d8128a8436881d297473e7e348bdfbd...,a51401f777a55b6ba3362828ee3d908ecc205c4f8d6984...,1,https://modeldb.science/119266?tab=2&file=CA1_...,https://modeldb.science/getModelFile?model=119...,TITLE K-A channel from Klee Ficker and Heinema...,None,119266,No PubMed link found
46,74,4ef920be6e5178c102f1b76d811a7868a5444b30602fc6...,a416d78093d04ec83478798645188447253dd4d66bb446...,3,https://modeldb.science/147141?tab=2&file=stdp...,https://modeldb.science/getModelFile?model=147...,": $Id: updown.mod,v 1.16 2009/02/16 22:56:52 b...",None,147141,No PubMed link found
...,...,...,...,...,...,...,...,...,...,...
706,965,a7947322212f7a1dc9065a7d188bf2358a0f46c6690100...,67b3461e300c60190e80c38dfe64599131739306fd09eb...,2,https://modeldb.science/147141?tab=2&file=stdp...,https://modeldb.science/getModelFile?model=147...,": $Id: sampen.mod,v 1.22 2010/01/21 22:39:19 s...",None,147141,No PubMed link found
711,970,0445f1f9d61b5b69fed4c58bb2a8f3dd2e5cbecf0edf12...,d75ed63d6367fe27c6e5671704594a510e8d83b57db835...,100,https://modeldb.science/184333?tab=2&file=4738...,https://modeldb.science/getModelFile?model=184...,: Reference: Colbert and Pan 2002\n\nNEURON\t{...,None,184333,No PubMed link found
724,985,eead06ca616245e9207104512dbeca0dfc5c6696bef40c...,1a9d58fc72e1fa85d2a2f0fae9f59fdbf34d273d2ad05b...,1,https://modeldb.science/139656?tab=2&file=netw...,https://modeldb.science/getModelFile?model=139...,NEURON {\n\tPOINT_PROCESS Gap\n\tPOINTER vnb\n...,None,139656,No PubMed link found
729,992,da680d20bdc0fbc6058b3821c49be15c552861cd696b67...,6ed7f324d6020d11df0ddbabbee6bad2b55403ea3f14b0...,1,https://modeldb.science/249463?tab=2&file=l5pc...,https://modeldb.science/getModelFile?model=249...,TITLE Low threshold calcium current\n:\n\nINDE...,None,249463,No PubMed link found


- 10% did not have abstracts

In [51]:
72/737

0.09769335142469471

In [52]:
!git add .
!git commit -m "added pubmed link"
!git push

[main 4ab0fad] added pubmed link
 4 files changed, 627 insertions(+), 790 deletions(-)
 rename code/{download_mod_files.ipynb => 1_download_mod_files.ipynb} (100%)
 create mode 100644 code/3_pubmed.ipynb
 delete mode 100644 code/preprocessed_new.csv
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 64 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 25.92 KiB | 1.08 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:innacohen/mod-extract.git
   ef77068..4ab0fad  main -> main
